# 🏭 ModelFlow Data Factory

This notebook generates synthetic e-commerce data and seeds it into:
1. **CSV files** in `seeds/` for `dbt seed`
2. **DuckDB tables** in `data/modelflow.duckdb` under the `bronze` schema

## Dataset Overview
| Entity     | Records | Description                          |
|------------|---------|--------------------------------------|
| Customers  | 1,000   | Consumer profiles with geo attributes |
| Products   | 50      | E-commerce catalog                   |
| Orders     | 2,000   | Transactional order records          |
| Shipments  | ~1,500  | Fulfillment / logistics records      |

In [ ]:
# ─────────────────────────────────────────
# Cell 1: Imports & configuration
# ─────────────────────────────────────────
import pandas as pd
import numpy as np
import duckdb
import random
import os
from pathlib import Path
from faker import Faker
from datetime import date, timedelta

# ── Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

# ── Config
NUM_CUSTOMERS = 1000
NUM_PRODUCTS  = 50
NUM_ORDERS    = 2000

# ── Paths (relative to project root)
PROJECT_ROOT  = Path().resolve()
SEEDS_DIR     = PROJECT_ROOT / 'seeds'
DATA_DIR      = PROJECT_ROOT / 'data'
DB_PATH       = DATA_DIR / 'modelflow.duckdb'

SEEDS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Seeds dir    : {SEEDS_DIR}')
print(f'DuckDB path  : {DB_PATH}')

In [ ]:
# ─────────────────────────────────────────
# Cell 2: Generate Customers
# ─────────────────────────────────────────
customers = pd.DataFrame([
    {
        'cust_id'    : i,
        'name'       : fake.name(),
        'email'      : fake.unique.email(),
        'country'    : fake.country(),
        'city'       : fake.city(),
        'segment'    : random.choice(['Consumer', 'Corporate', 'Home Office']),
        'created_at' : fake.date_between(start_date='-2y', end_date='-6m'),
        'updated_at' : fake.date_between(start_date='-6m', end_date='today'),
    }
    for i in range(1, NUM_CUSTOMERS + 1)
])

print(f'Customers: {len(customers):,} rows')
customers.head(3)

In [ ]:
# ─────────────────────────────────────────
# Cell 3: Generate Products
# ─────────────────────────────────────────
CATEGORY_PRICES = {
    'Electronics': (50,  500),
    'Apparel'    : (10,  150),
    'Home'       : (20,  300),
    'Toys'       : (5,   100),
    'Sports'     : (15,  250),
}

products_list = []
for i in range(1, NUM_PRODUCTS + 1):
    category = random.choice(list(CATEGORY_PRICES.keys()))
    lo, hi   = CATEGORY_PRICES[category]
    products_list.append({
        'prod_id'      : i,
        'product_name' : fake.catch_phrase(),
        'category'     : category,
        'subcategory'  : fake.word().capitalize(),
        'price'        : round(random.uniform(lo, hi), 2),
        'cost'         : round(random.uniform(lo * 0.4, hi * 0.6), 2),
        'sku'          : fake.bothify('??-####').upper(),
        'is_active'    : random.choice([True, True, True, False]),
    })

products = pd.DataFrame(products_list)
print(f'Products: {len(products):,} rows')
products.head(3)

In [ ]:
# ─────────────────────────────────────────
# Cell 4: Generate Orders
# ─────────────────────────────────────────
orders_list = []
for i in range(1, NUM_ORDERS + 1):
    order_date = fake.date_between(start_date='-1y', end_date='today')
    orders_list.append({
        'order_id'         : i,
        'cust_id'          : random.randint(1, NUM_CUSTOMERS),
        'prod_id'          : random.randint(1, NUM_PRODUCTS),
        'order_date'       : order_date,
        'quantity'         : random.randint(1, 10),
        'discount_pct'     : round(random.choice([0, 0, 0, 5, 10, 15, 20]), 2),
        'status'           : random.choice(['Shipped', 'Shipped', 'Cancelled', 'Pending', 'Returned']),
        'channel'          : random.choice(['Web', 'Mobile', 'In-Store', 'Partner']),
        'payment_method'   : random.choice(['Credit Card', 'PayPal', 'Bank Transfer', 'Crypto']),
    })

orders_df = pd.DataFrame(orders_list)
print(f'Orders: {len(orders_df):,} rows')
orders_df.head(3)

In [ ]:
# ─────────────────────────────────────────
# Cell 5: Generate Shipments
# ─────────────────────────────────────────
CARRIERS = ['DHL', 'FedEx', 'UPS', 'USPS', 'DPD']

# Only shipped & returned orders get a shipment record
shipped_orders = orders_df[
    orders_df['status'].isin(['Shipped', 'Returned'])
].copy()

shipments_list = []
for idx, (_, row) in enumerate(shipped_orders.iterrows(), start=1):
    order_date = row['order_date']
    if isinstance(order_date, str):
        order_date = date.fromisoformat(order_date)
    ship_date     = order_date + timedelta(days=random.randint(1, 5))
    delivery_date = ship_date + timedelta(days=random.randint(1, 7))
    shipments_list.append({
        'shipment_id'       : idx,
        'order_id'          : row['order_id'],
        'carrier'           : random.choice(CARRIERS),
        'tracking_number'   : fake.bothify('??########').upper(),
        'ship_date'         : ship_date,
        'delivery_date'     : delivery_date,
        'weight_kg'         : round(random.uniform(0.1, 20.0), 2),
        'shipping_cost'     : round(random.uniform(3.0, 50.0), 2),
        'status'            : 'Delivered' if row['status'] == 'Shipped' else 'Returned',
    })

shipments_df = pd.DataFrame(shipments_list)
print(f'Shipments: {len(shipments_df):,} rows')
shipments_df.head(3)

In [ ]:
# ─────────────────────────────────────────
# Cell 6: Export CSVs to seeds/
# ─────────────────────────────────────────
csv_exports = {
    'raw_customers' : customers,
    'raw_products'  : products,
    'raw_orders'    : orders_df,
    'raw_shipments' : shipments_df,
}

for name, df in csv_exports.items():
    path = SEEDS_DIR / f'{name}.csv'
    df.to_csv(path, index=False)
    print(f'✅  Exported {len(df):>6,} rows → {path.relative_to(PROJECT_ROOT)}')

In [ ]:
# ─────────────────────────────────────────
# Cell 7: Persist to DuckDB bronze schema
# ─────────────────────────────────────────
con = duckdb.connect(str(DB_PATH))

# Create bronze schema
con.execute('CREATE SCHEMA IF NOT EXISTS bronze')

bronze_tables = {
    'bronze.raw_customers' : customers,
    'bronze.raw_products'  : products,
    'bronze.raw_orders'    : orders_df,
    'bronze.raw_shipments' : shipments_df,
}

for table_name, df in bronze_tables.items():
    con.execute(f'CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM df')
    count = con.execute(f'SELECT COUNT(*) FROM {table_name}').fetchone()[0]
    print(f'✅  Created {table_name:<30} → {count:,} rows')

con.close()
print(f'\n🦆 DuckDB database ready at: {DB_PATH}')

In [ ]:
# ─────────────────────────────────────────
# Cell 8: Quick validation
# ─────────────────────────────────────────
con = duckdb.connect(str(DB_PATH), read_only=True)

print('=== Schema: bronze ===')
tables = con.execute("SELECT table_name, estimated_size FROM duckdb_tables() WHERE schema_name='bronze'").df()
display(tables)

print('\n=== Sample: bronze.raw_orders ===')
display(con.execute('SELECT * FROM bronze.raw_orders LIMIT 5').df())

print('\n=== Orders by status ===')
display(con.execute("""
    SELECT status, COUNT(*) as cnt, ROUND(SUM(quantity),0) as total_qty
    FROM bronze.raw_orders
    GROUP BY 1
    ORDER BY 2 DESC
""").df())

con.close()